# Fake News Detection using FLAN-T5
## Notebook 3: Zero-Shot vs Few-Shot Prompting

### Course
ADS-509 – Applied Large Language Models for Data Science

### Team Project

### Objective

This notebook evaluates the performance of the pre-trained FLAN-T5 model on a Fake News Detection task before fine-tuning.

Two prompting strategies are compared:

- Zero-Shot Prompting
- Few-Shot Prompting

The results obtained here serve as the baseline for comparison with the fine-tuned model developed in the next notebook.

In [38]:
pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.


In [40]:
pip install transformers datasets evaluate sentencepiece accelerate

Note: you may need to restart the kernel to use updated packages.


In [42]:
import pandas as pd
import numpy as np

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")

In [44]:
import requests

requests.get("https://huggingface.co")

<Response [200]>

In [46]:
import ssl
import certifi

print(ssl.OPENSSL_VERSION)
print(certifi.where())

OpenSSL 3.4.1 11 Feb 2025
C:\Users\anish\anaconda3\Lib\site-packages\certifi\cacert.pem


In [48]:
import transformers

print(transformers.__version__)

5.12.1


In [52]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cpu


In [54]:
import huggingface_hub
import httpx

print("huggingface_hub:", huggingface_hub.__version__)
print("httpx:", httpx.__version__)

huggingface_hub: 1.20.1
httpx: 0.27.0


In [56]:
import httpx

r = httpx.get("https://huggingface.co")
print(r.status_code)

200


In [58]:
import os

print(os.path.getsize("clean_news.csv"))

114266267


In [60]:
with open("clean_news.csv", "rb") as f:
    print(f.read(100))

b'"content","label"\r\n"WOW! LEFTIST LIBRARIAN REJECTS Shipment Of Children\xe2\x80\x99s Books Donated By Melania'


In [62]:
import os

print(os.listdir())

['.ipynb_checkpoints', '01_Data_Preprocessing.ipynb', '02_Exploratory_Data_Analysis.ipynb', '03_ZeroShot_FewShot.ipynb', '04_FLAN_T5_FineTuning.ipynb', 'clean_news.csv', 'figures']


In [64]:
import os

print(os.path.getsize("clean_news.csv"))

114266267


In [66]:
import csv

count = 0

with open("clean_news.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)

    try:
        for row in reader:
            count += 1
    except Exception as e:
        print("Stopped at row:", count)
        print(e)

print("Rows successfully read:", count)

Rows successfully read: 44690


In [68]:
import pandas as pd

for i, chunk in enumerate(pd.read_csv("clean_news.csv", chunksize=5000)):
    print(f"Chunk {i+1}: {chunk.shape}")

Chunk 1: (5000, 2)
Chunk 2: (5000, 2)
Chunk 3: (5000, 2)
Chunk 4: (5000, 2)
Chunk 5: (5000, 2)
Chunk 6: (5000, 2)
Chunk 7: (5000, 2)
Chunk 8: (5000, 2)
Chunk 9: (4689, 2)


In [70]:
import pandas as pd

try:
    df = pd.read_csv(
        "clean_news.csv",
        engine="python",
        on_bad_lines="skip"
    )
    print(df.shape)
except Exception as e:
    print(e)

(44689, 2)


In [72]:
df = pd.read_csv("clean_news.csv")
df.shape

(44689, 2)

In [74]:
import pandas as pd

test = pd.read_csv("clean_news.csv", nrows=5)

test

,content,label
0,WOW! LEFTIST LIBRARIAN REJECTS Shipment Of Chi...,0
1,Kenya opposition leader calls for calm in slum...,1
2,Egypt rejects U.S. decision to move its embass...,1
3,(AUDIO)NATION OF ISLAM LEADER FARRAKHAN: “WE W...,0
4,Trump Rally Nearly Turns Into A Full-Blown Ra...,0


In [76]:
df = pd.read_csv("clean_news.csv")

df.shape

(44689, 2)

In [78]:
import pandas as pd

df = pd.read_csv(
    "clean_news.csv",
    engine="python",
    encoding="utf-8"
).shape

In [79]:
df = pd.read_csv(
    "clean_news.csv",
    engine="python"
)
df.head()

,content,label
0,WOW! LEFTIST LIBRARIAN REJECTS Shipment Of Chi...,0
1,Kenya opposition leader calls for calm in slum...,1
2,Egypt rejects U.S. decision to move its embass...,1
3,(AUDIO)NATION OF ISLAM LEADER FARRAKHAN: “WE W...,0
4,Trump Rally Nearly Turns Into A Full-Blown Ra...,0


In [81]:
df = pd.read_csv(
    "clean_news.csv",
    engine="python",
    encoding="utf-8"
)

df.head()

,content,label
0,WOW! LEFTIST LIBRARIAN REJECTS Shipment Of Chi...,0
1,Kenya opposition leader calls for calm in slum...,1
2,Egypt rejects U.S. decision to move its embass...,1
3,(AUDIO)NATION OF ISLAM LEADER FARRAKHAN: “WE W...,0
4,Trump Rally Nearly Turns Into A Full-Blown Ra...,0


In [83]:
print("Dataset Shape:", df.shape)

print("\nClass Distribution:")

print(df["label"].value_counts())

Dataset Shape: (44689, 2)

Class Distribution:
label
0    23478
1    21211
Name: count, dtype: int64


In [86]:
sample_df = df.sample(
    n=50,
    random_state=42
).reset_index(drop=True)

sample_df.head()

,content,label
0,IMMIGRANT Ghanaian Woman Pleads Guilty To $3.6...,0
1,"WHOA! RAND PAUL, NEWT GINGRICH RIP OBAMA’S Nat...",0
2,WATCH: Hilariously Stupid Trump Fans Fooled B...,0
3,Trump says has 'total confidence' in Attorney ...,1
4,Mexico says won't pay for Trump's 'terrible' b...,1


In [88]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model.to(device)

print("Model Loaded Successfully!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model Loaded Successfully!


In [90]:
def zero_shot_predict(text):

    prompt = f"""
Classify the following news article.

Answer only with:

Fake

or

Real

News:

{text}
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=5
    )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return prediction.strip()

In [92]:
article = sample_df.loc[0, "content"]

prediction = zero_shot_predict(article)

print(prediction)

Fake


In [94]:
def convert_prediction(pred):

    pred = pred.lower()

    if "fake" in pred:
        return 0

    if "real" in pred:
        return 1

    return -1

In [96]:
predictions = []

for article in sample_df["content"]:

    pred = zero_shot_predict(article)

    pred = convert_prediction(pred)

    predictions.append(pred)

sample_df["prediction"] = predictions

sample_df.head()

,content,label,prediction
0,IMMIGRANT Ghanaian Woman Pleads Guilty To $3.6...,0,0
1,"WHOA! RAND PAUL, NEWT GINGRICH RIP OBAMA’S Nat...",0,0
2,WATCH: Hilariously Stupid Trump Fans Fooled B...,0,0
3,Trump says has 'total confidence' in Attorney ...,1,0
4,Mexico says won't pay for Trump's 'terrible' b...,1,0


In [97]:
accuracy = accuracy_score(
    sample_df["label"],
    sample_df["prediction"]
)

print(f"Zero-Shot Accuracy: {accuracy:.2%}")

Zero-Shot Accuracy: 50.00%


In [100]:
print(

classification_report(

sample_df["label"],

sample_df["prediction"]

)

)

              precision    recall  f1-score   support

           0       0.50      1.00      0.67        25
           1       0.00      0.00      0.00        25

    accuracy                           0.50        50
   macro avg       0.25      0.50      0.33        50
weighted avg       0.25      0.50      0.33        50



In [102]:
cm = confusion_matrix(
    sample_df["label"],
    sample_df["prediction"]
)

print(cm)

[[25  0]
 [25  0]]


In [104]:
few_shot_examples = """
Example 1

News:
Donald Trump signed a secret agreement with aliens.

Answer:
Fake

Example 2

News:
Reuters reported that the Federal Reserve kept interest rates unchanged.

Answer:
Real
"""

In [106]:
def few_shot_predict(text):

    prompt = f"""
{few_shot_examples}

Now classify this news article.

Answer only with:

Fake

or

Real

News:

{text}
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=5
    )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return prediction.strip()

In [108]:
few_predictions = []

for article in sample_df["content"]:

    pred = few_shot_predict(article)

    pred = convert_prediction(pred)

    few_predictions.append(pred)

sample_df["few_prediction"] = few_predictions

In [109]:
few_accuracy = accuracy_score(
    sample_df["label"],
    sample_df["few_prediction"]
)

print(f"Few-Shot Accuracy: {few_accuracy:.2%}")

Few-Shot Accuracy: 8.00%


In [110]:
comparison = pd.DataFrame({

    "Method":[
        "Zero-Shot",
        "Few-Shot"
    ],

    "Accuracy":[
        accuracy,
        few_accuracy
    ]

})

comparison

,Method,Accuracy
0,Zero-Shot,0.50
1,Few-Shot,0.08


# Observations

The pre-trained FLAN-T5 model was evaluated using both zero-shot and few-shot prompting before fine-tuning.

### Zero-Shot Prompting

Without any examples, the model achieved an accuracy of approximately 50% on the sampled news articles. While the model was able to classify some articles correctly, the results indicate that the general-purpose FLAN-T5 model struggles to reliably distinguish between fake and real news without task-specific training.

### Few-Shot Prompting

Contrary to expectations, the few-shot prompting approach achieved a lower accuracy of approximately 8%. The small set of manually created examples did not generalize well to the dataset and appeared to bias the model toward incorrect predictions. This demonstrates that the effectiveness of few-shot prompting depends heavily on the quality and representativeness of the provided examples.

### Conclusion

For this dataset, zero-shot prompting performed better than the chosen few-shot prompting strategy. These results establish a useful baseline before fine-tuning the FLAN-T5 model. In the next notebook, supervised fine-tuning will be performed to improve classification performance on the Fake and Real News dataset.